In [1]:
import os
import cv2
import json
# from io import BytesIO
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import scipy
import cv2
from skimage.measure import regionprops, label, shannon_entropy
from skimage.transform import resize
from skimage.color import rgb2gray
from skimage import img_as_ubyte
from brisque import BRISQUE
from skvideo.measure import niqe
from pypiqe import piqe
from mahotas.features import zernike_moments
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import transforms
import timm
import segmentation_models_pytorch as smp

# Patch imresize if missing
if not hasattr(scipy.misc, "imresize"):
    def imresize(arr, size, interp=None, mode=None):
        if isinstance(size, float):  # scale factor
            new_shape = (int(arr.shape[0] * size), int(arr.shape[1] * size))
        else:
            new_shape = size[:2]
        arr_resized = resize(arr, new_shape, order=3, mode="reflect", anti_aliasing=True)
        arr_resized = (arr_resized * 255).astype(np.uint8)
        return arr_resized
    scipy.misc.imresize = imresize

# Patch for deprecated NumPy aliases (for backward compatibility)
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'bool'):
    np.bool = bool

In [2]:
# Hyperparameters and paths
DATA_DIR = r'/home/c/choton/beemachine/datasets/Others/CUB_200_2011/'
NUM_PARTS = 12      # Including background
CLASSIFIER_NAME = "convnext_nano.in12k"
SEGMENTATION_NAME = "deeplabv3plus"
SEGMENTATION_ENCODER = "resnext50_32x4d"
CLASSIFIER_WEIGHTS = r"/home/c/choton/beemachine/codes/AG_vision_2026/1_classification/CUB/convnext_nano.in12k_timm_logs/checkpoints/best_model.pth"
SEGMENTATION_WEIGHTS = r"/home/c/choton/beemachine/codes/AG_vision_2026/2_segmentation/CUB/deeplabv3plus/lightning_logs/version_2/checkpoints/epoch=199-step=13400.ckpt"
# SHAPE_FEATS_PATH = r"/home/c/choton/beemachine/codes/everyday_test/November_25/Nov12_25/shape_features/beemachine_small_2025_part_features.csv"
# FEATURE_SIZE = 937

# Model training hyperparameters
MODEL = "arch2_cub_convnext_nano"
DEVICE_ID = 0
BATCH_SIZE = 128
IMAGE_SIZE = 224
EPOCHS = 100
LR = 1e-4
OPTIMIZER = "AdamW"
LOG_DIR = f"./{MODEL}_logs/"
SEED = 42

# Seeding for consistant results
torch.manual_seed(SEED)
np.random.seed(SEED)

In [3]:
# Create SIFT and ORB detectors once
sift = cv2.SIFT_create()
orb = cv2.ORB_create()
bri_obj = BRISQUE(url=False)

def extract_base_features(mask):
    """Compute geometric, Zernike, Fourier, and texture shape descriptors from a binary mask."""
    
    features = ["area", "perimeter", "aspect_ratio", "extent", "solidity", "eccentricity", 
        "orientation", "circularity", "elongation", "compactness"]
    
    if mask is None or mask.sum() == 0:
        return {f: 0 for f in features}

    # --- Region properties ---
    # mask = mask.astype(np.uint8)
    labeled = label(mask)
    props = regionprops(labeled)
    if len(props) == 0:
        return {f: 0 for f in features}
    p = props[0]
    major_axis = p.major_axis_length
    minor_axis = p.minor_axis_length

    # ----- base shape features -----
    area = p.area
    perimeter = max(p.perimeter, 1e-6) # Ignoring too small perimeters
    aspect_ratio = major_axis / minor_axis if minor_axis > 0 else 0 # L_major / L_minor
    extent = p.extent
    solidity = p.solidity
    eccentricity = p.eccentricity
    orientation = p.orientation
    circularity = 4 * np.pi * area / (perimeter ** 2)
    elongation = 1 - (minor_axis / major_axis) if major_axis > 0 else 0
    # convexity = p.perimeter_convex / perimeter
    compactness = (perimeter ** 2) / (4 * np.pi * area + 1e-6)

    # ----- Assemble features -----
    features_d = {
        "area": area,
        "perimeter": perimeter,
        "aspect_ratio": aspect_ratio,
        "extent": extent,
        "solidity": solidity,
        "eccentricity": eccentricity,
        "orientation": orientation,
        "circularity": circularity,
        "elongation": elongation,
        "compactness": compactness
    }
    return features_d

def compute_sift_features(image, mask=None):
    # if isinstance(image, torch.Tensor):
    #     image = image.detach().cpu().numpy().transpose(1, 2, 0) # Transform tensor to numpy image
    # if isinstance(mask, torch.Tensor):
    #     mask = mask.numpy().astype(np.uint8)
    gray= cv2.cvtColor(img_as_ubyte(image), cv2.COLOR_RGB2GRAY) # converts image into uint8 and mask as input
    keypoints, descriptors = sift.detectAndCompute(gray, mask)
    if descriptors is None:
        descriptors = np.full((0, 128), np.nan, dtype=np.float32)  # return empty array if no keypoints
    return keypoints, descriptors

def compute_orb_features(image, mask=None):
    # if isinstance(image, torch.Tensor):
    #     image = image.detach().cpu().numpy().transpose(1, 2, 0) # Transform tensor to numpy image
    # if isinstance(mask, torch.Tensor):
    #     mask = mask.numpy().astype(np.uint8)
    gray= cv2.cvtColor(img_as_ubyte(image), cv2.COLOR_RGB2GRAY)
    keypoints, descriptors = orb.detectAndCompute(gray, mask)
    if descriptors is None:
        descriptors = np.full((0, 32), np.nan, dtype=np.float32)  # return empty array if no keypoints
    return keypoints, descriptors

def compute_hu_moments(mask):
    # if not isinstance(mask, np.ndarray):
    #     mask = mask.numpy().astype(np.uint8)
    moments = cv2.moments(mask)
    hu = cv2.HuMoments(moments).flatten()
    hu = np.log(np.abs(hu) + 1e-12) # log-scale for stability
    return hu

def compute_zernike_moments(mask, degree=8):
    # if not isinstance(mask, np.ndarray):
    #     mask = mask.numpy().astype(np.uint8)
    radius = min(mask.shape) // 2
    mask_norm = mask / mask.max() if mask.max() > 0 else mask
    zern = zernike_moments(mask_norm, radius=radius, degree=degree)
    return zern

# *** Updated fourier descriptors (Dec 4, 2025)
def compute_fourier_descriptors(mask, image=None, fourier_harmonics=20, visualize=False):
    if not isinstance(mask, np.ndarray): # Ensure proper mask format
        mask = mask.numpy().astype(np.uint8)
    # --- 2. Find largest contour (object part) ---
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not contours:
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)
    cnt = max(contours, key=cv2.contourArea)
    if len(cnt) < 3:
        # Too few points for Fourier transform
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)
    
    # Translation invariance: center contour
    complex_contour = cnt[:,0,0] + 1j * cnt[:,0,1]
    fd = np.fft.fft(complex_contour)
    
    if visualize: # ** IMPORTANT: Visualization uses raw contour (so you can see the real shape), descriptors are centered.
        # Convert image if needed
        H, W = mask.shape
        if image is not None:
            if isinstance(image, torch.Tensor):
                image = image.detach().cpu().numpy().transpose(1, 2, 0)
            elif isinstance(image, Image.Image):
                image = np.array(image.convert('RGB'))
            elif image.dtype != np.uint8:  # NumPy float → uint8
                image = (image*255).astype(np.uint8)
            img_draw = image.copy()
        else:
            img_draw = np.zeros((H, W, 3), dtype=np.uint8)
        cv2.drawContours(img_draw, [cnt.astype(np.int32)], -1, (0, 255, 0), 2)

        fd_recon = fd.copy()
        keep = fourier_harmonics
        if 2 * keep < len(fd_recon):
            fd_recon[keep:-keep] = 0 # Safe zeroing
        else:
            fd_recon[keep:] = 0
        recon = np.fft.ifft(fd_recon)
        pts = np.column_stack((recon.real, recon.imag)).astype(np.int32)

        for i in range(len(pts)):
            cv2.line(img_draw, tuple(pts[i]), tuple(pts[(i + 1) % len(pts)]), (255, 0, 255), 1)
        plt.figure(figsize=(16, 6))
        plt.imshow(img_draw)
        plt.axis('off')
        plt.title("Shape Descriptors Overlay")
        plt.legend(
            handles=[
                Patch(facecolor='green', edgecolor='green'),
                Patch(facecolor='magenta', edgecolor='magenta')
            ],
            labels=["Contour", "Fourier Reconstruction"],
            loc='upper right'
        )
        plt.show()
    
    cnt_centered = complex_contour - np.mean(complex_contour)
    fd = np.fft.fft(cnt_centered)
    if len(fd) < 2 or np.abs(fd[1]) == 0:
        return np.full(fourier_harmonics, np.nan, dtype=np.float32)

    # Scale invariance: divide by first descriptor magnitude
    fd = fd / np.abs(fd[1])

    # Rotation invariance: use only magnitudes
    fd_normalized = np.abs(fd)

    # Keep only first N harmonics
    fd_truncated = fd_normalized[:fourier_harmonics]
    if len(fd_truncated) < fourier_harmonics:
        fd_truncated = np.concatenate([fd_truncated, np.full((fourier_harmonics - len(fd_truncated)), np.nan)])
    return fd_truncated

def extract_shape_features(image, mask):
    # Compute base features
    features = extract_base_features(mask)

    # Compute sift features
    sift_kp, sift_ds = compute_sift_features(image, mask)
    sift_sizes = [k.size for k in sift_kp]
    if sift_ds.shape[0] > 0:
        sift_mean_ds = np.nanmean(sift_ds, axis=0)
    else:
        sift_mean_ds = np.full(128, np.nan)
    sift_dict = {f'sift_ds{i+1}': sift_mean_ds[i] for i in range(len(sift_mean_ds))}
    sift_dict['sift_kp_n'] = len(sift_kp)
    sift_dict['sift_kp_size'] = np.max(sift_sizes) if sift_sizes else 0

    # Compute orb features
    orb_kp, orb_ds = compute_orb_features(image, mask)
    if orb_ds.shape[0] > 0:
        orb_mean_ds = np.nanmean(orb_ds, axis=0)
    else:
        orb_mean_ds = np.full(32, np.nan)
    orb_dict = {f'orb_ds{i+1}': orb_mean_ds[i] for i in range(len(orb_mean_ds))}
    orb_dict['orb_kp_n'] = len(orb_kp)

    # Compute hu moments
    hu_moments = compute_hu_moments(mask)
    hu_dict = {f"hu{i+1}": hu_moments[i] for i in range(len(hu_moments))}

    # Compute Zernike moments
    zern_moments = compute_zernike_moments(mask, degree=8)
    zern_dict = {f"zernike_{i+1}": zern_moments[i] for i in range(len(zern_moments))}

    # Compute fourier descriptors
    fourier_ds = compute_fourier_descriptors(mask, fourier_harmonics=20)
    fourier_dict = {f"fourier_{i+1}": fourier_ds[i] for i in range(len(fourier_ds))}

    features.update(sift_dict)
    features.update(orb_dict)
    features.update(hu_dict)
    features.update(zern_dict)
    features.update(fourier_dict)
    converted = {k: np.float32(v) for k, v in features.items()}
    return converted

def extract_visual_features(image, mask):
    # --- 1. Ensure binary uint8 mask ---
    if not isinstance(mask, np.ndarray):
        mask = mask.numpy().astype(np.uint8)
    # Convert image to numpy
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy().transpose(1, 2, 0)
    elif isinstance(image, Image.Image):
        image = np.array(image.convert('RGB'))
    img_cropped = np.zeros_like(image)
    img_cropped[mask==1] = image[mask==1]
    # plt.imshow(img_cropped)

    # --- Brightness ---
    brightness = np.mean(img_cropped)

    # --- Contrast (standard deviation of luminance) ---
    gray = rgb2gray(img_cropped)
    contrast = np.std(gray)

    # --- Sharpness (variance of Laplacian) ---
    gray_8u = (gray * 255).astype(np.uint8)
    lap_var = cv2.Laplacian(gray_8u, cv2.CV_64F).var()

    # --- Colorfulness (Hasler & Süsstrunk, 2003) ---
    (R, G, B) = cv2.split(img_cropped)
    rg = np.abs(R - G)
    yb = np.abs(0.5 * (R + G) - B)
    std_rg, std_yb = np.std(rg), np.std(yb)
    mean_rg, mean_yb = np.mean(rg), np.mean(yb)
    colorfulness = np.sqrt(std_rg**2 + std_yb**2) + 0.3 * np.sqrt(mean_rg**2 + mean_yb**2)

    # --- Entropy (texture complexity) ---
    entropy = shannon_entropy(gray)

    # BRISQUE
    bri_obj = BRISQUE(url=False)
    brisque_score = bri_obj.score(img_cropped)

    # NIQE
    niqe_score = niqe(gray)

    # PIQE
    piqe_score, activityMask, noticeableArtifactMask, noiseMask = piqe(gray)

    # --- Aggregate descriptors ---
    descriptors = {
        "brightness": np.float32(brightness),
        "contrast": np.float32(contrast),
        "sharpness": np.float32(lap_var),
        "colorfulness": np.float32(colorfulness),
        "entropy": np.float32(entropy),
        "brisque": np.float32(brisque_score),
        "niqe": np.float32(niqe_score.item()),
        "piqe": np.float32(piqe_score)
    }
    return descriptors

def extract_combined_features(image, mask): 
    # ---- Convert once ----
    if isinstance(image, torch.Tensor):
        image = image.detach().cpu().numpy().transpose(1, 2, 0)
    if isinstance(image, Image.Image):
        image = np.asarray(image)
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy().astype(np.uint8)
    mask_u8 = mask.astype(np.uint8)
    image_f = (image - image.min()) / (image.max() - image.min() + 1e-8) # Linearly rescale to [0, 1] and avoid division by zero
    image_u8 = img_as_ubyte(image_f)

    combined_features = extract_shape_features(image_u8, mask_u8)
    vis_features = extract_visual_features(image_f, mask_u8)
    combined_features.update(vis_features)
    return combined_features

def extract_all_features(image, mask):
    """
    Generalized feature extraction for N body parts.
    mask: 2D array where each unique integer >0 is a body part ID.
    """

    # Get unique body-part IDs (excluding background 0)
    part_ids = sorted([pid for pid in np.unique(mask) if pid > 0])

    # Extract features for each part
    part_features = {}
    for pid in part_ids:
        part_mask = mask == pid
        feats = extract_combined_features(image, part_mask)
        part_features[pid] = feats

    # Full body features
    full_mask = mask > 0
    full_feats = extract_combined_features(image, full_mask)

    # Compute area sums
    areas = {pid: part_features[pid]["area"] for pid in part_ids}
    total_area = sum(areas.values()) + 1e-6

    # ----------- Ratio Features -----------
    # Pairwise ratios: area_i / area_j
    ratio_feats = {}
    for i in part_ids:
        for j in part_ids:
            if i != j:
                ratio_feats[f"area_ratio_part{i}_to_part{j}"] = areas[i] / (areas[j] + 1e-6)

    # Area-to-total ratios
    for pid in part_ids:
        ratio_feats[f"area_ratio_part{pid}_to_total"] = areas[pid] / total_area

    # ----------- Build Record -----------
    record = {}

    # Add each part's features
    for pid in part_ids:
        feats = part_features[pid]
        for k, v in feats.items():
            record[f"part{pid}_{k}"] = v

    # Add full-body features
    for k, v in full_feats.items():
        record[f"full_{k}"] = v

    # Add ratio features
    record.update(ratio_feats)
    return record

In [4]:
# Load classifier
def load_classifier(name, weights, num_classes, device_id):
    # Define the classification model and load weights
    model = timm.create_model(name, pretrained=False, num_classes=num_classes).to(f"cuda:{device_id}")
    state_dict_c = torch.load(weights, map_location=f"cuda:{device_id}")
    model.load_state_dict(state_dict_c) # Load the classifier state dict
    return model

# Load segmentation model
def load_segmentation(name, encoder_name, checkpoint_path):    
    # Define the segmentation model and load weights
    model = smp.create_model(
                name,
                encoder_name=encoder_name,
                in_channels = 3, # 3 for RGB images
                classes = 12
            ).to(f"cuda:{DEVICE_ID}")
    # checkpoint_path = r"/home/c/choton/beemachine/codes/everyday_test/oct1_25/segmentation_models/lightning_logs/fpn/lightning_logs/version_0/checkpoints/epoch=199-step=37400.ckpt"
    checkpoint = torch.load(checkpoint_path, map_location=rf"cuda:{DEVICE_ID}")

    # Sometimes Lightning adds "state_dict"
    if "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]

        # Remove "model." or "net." prefixes if present
        new_state_dict = {k.replace("model.", "").replace("net.", ""): v 
                        for k, v in state_dict.items() if k not in ["mean", "std"]}

        model.load_state_dict(new_state_dict)
    else:
        model.load_state_dict(checkpoint)
    # model.to(f"cuda:{DEVICE_ID}")
    return model

#  Segmentation prediction
def predict_mask(model, image_tensor):
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.to(f"cuda:{DEVICE_ID}")
        output = model(image_tensor.unsqueeze(0))
        if isinstance(output, dict):  # handle SMP model dict outputs
            output = output['out']
        probs = torch.softmax(output, dim=1)
        mask = torch.argmax(probs, dim=1).squeeze(0).cpu().numpy()
    return mask  # (H, W) integer mask

def compute_loss(outputs, targets, mask_logits=None, mask_gt=None):
    cls_loss = torch.nn.functional.cross_entropy(outputs, targets)
    seg_loss = 0
    if mask_gt is not None:
        seg_loss = torch.nn.functional.binary_cross_entropy_with_logits(mask_logits, mask_gt)
    total_loss = cls_loss + 0.5 * seg_loss
    return total_loss

In [5]:
# =========================
# 2. LOAD OFFICIAL SPLITS
# =========================

images_df = pd.read_csv(
    os.path.join(DATA_DIR, "images.txt"),
    sep=" ",
    names=["img_id", "filepath"]
)

labels_df = pd.read_csv(
    os.path.join(DATA_DIR, "image_class_labels.txt"),
    sep=" ",
    names=["img_id", "label"]
)

split_df = pd.read_csv(
    os.path.join(DATA_DIR, "train_test_split.txt"),
    sep=" ",
    names=["img_id", "is_train"]
)

# Merge all
df = images_df.merge(labels_df, on="img_id")
df = df.merge(split_df, on="img_id")

# Convert labels to 0-index
df["label"] = df["label"] - 1

# =========================
# 3. CREATE 75:15:10 SPLIT
# =========================

# First separate official test
official_train = df[df["is_train"] == 1]
official_test = df[df["is_train"] == 0]

# Combine all data
full_df = pd.concat([official_train, official_test])

# 75% train, 25% temp
train_df, temp_df = train_test_split(
    full_df,
    test_size=0.25,
    stratify=full_df["label"],
    random_state=SEED
)

# Split temp into val (15%) and test (10%)
val_ratio = 0.15 / (0.15 + 0.10)

val_df, test_df = train_test_split(
    temp_df,
    test_size=(1 - val_ratio),
    stratify=temp_df["label"],
    random_state=SEED
)

print(f"Train: {len(train_df)}, Val:   {len(val_df)}, Test:  {len(test_df)}")
full_df

Train: 8841, Val:   1768, Test:  1179


,img_id,filepath,label,is_train
1,2,001.Black_footed_Albatross/Black_Footed_Albatr...,0,1
3,4,001.Black_footed_Albatross/Black_Footed_Albatr...,0,1
4,5,001.Black_footed_Albatross/Black_Footed_Albatr...,0,1
6,7,001.Black_footed_Albatross/Black_Footed_Albatr...,0,1
7,8,001.Black_footed_Albatross/Black_Footed_Albatr...,0,1
...,...,...,...,...
11779,11780,200.Common_Yellowthroat/Common_Yellowthroat_00...,199,0
11782,11783,200.Common_Yellowthroat/Common_Yellowthroat_00...,199,0
11784,11785,200.Common_Yellowthroat/Common_Yellowthroat_00...,199,0
11785,11786,200.Common_Yellowthroat/Common_Yellowthroat_00...,199,0


In [6]:
class CUBDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(DATA_DIR, "images", row["filepath"])
        image = Image.open(img_path).convert("RGB")
        label = row["label"]
        if self.transform:
            image = self.transform(image)
        return image, label, idx

In [7]:
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = CUBDataset(train_df, train_transform)
val_dataset = CUBDataset(val_df, val_transform)
test_dataset = CUBDataset(test_df, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, persistent_workers=True, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, persistent_workers=True, pin_memory=True)

classes = set(labels_df['label'])
num_classes = len(classes)
print(f"Classes: {num_classes} | Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

Classes: 200 | Train: 8841 | Val: 1768 | Test: 1179


In [9]:
all_feats_path = r'/home/c/choton/beemachine/codes/AG_vision_2026/4_shape_feature_analysis/CUB/shape_features_cub/cub_shape_features.csv'
all_feats_df = pd.read_csv(all_feats_path, index_col=False)
all_feats_df

,image,class_id,part1_area,part1_perimeter,part1_aspect_ratio,part1_extent,part1_solidity,part1_eccentricity,part1_orientation,part1_circularity,...,area_ratio_part9_to_part4,area_ratio_part9_to_part10,area_ratio_part10_to_part8,area_ratio_part10_to_part9,area_ratio_part3_to_part8,area_ratio_part8_to_part3,area_ratio_part10_to_part11,area_ratio_part11_to_part10,area_ratio_part3_to_part9,area_ratio_part9_to_part3
0,Black_Footed_Albatross_0001_796111.jpg,0,1733.0,2.802498e+02,2.451833,0.396205,0.636197,0.913045,-0.981756,2.772796e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Black_Footed_Albatross_0002_55.jpg,0,3173.0,3.209716e+02,1.939136,0.436571,0.685017,0.856773,-0.954378,3.870319e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Black_Footed_Albatross_0003_796136.jpg,0,1484.0,2.132670e+02,3.039578,0.607699,0.745729,0.944332,1.424403,4.100118e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Black_Footed_Albatross_0005_796090.jpg,0,3035.0,4.076762e+02,1.116284,0.438837,0.622692,0.444399,-0.610094,2.294763e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Black_Footed_Albatross_0006_796065.jpg,0,1874.0,2.030538e+02,2.810191,0.648892,0.876110,0.934544,-1.383102,5.711591e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11783,Common_Yellowthroat_0118_190805.jpg,199,3405.0,3.869899e+02,2.159456,0.415751,0.718809,0.886317,-0.405297,2.857115e-01,...,5.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11784,Common_Yellowthroat_0121_190597.jpg,199,3523.0,3.412203e+02,1.904097,0.553670,0.769382,0.850989,-0.319369,3.802355e-01,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11785,Common_Yellowthroat_0122_190570.jpg,199,987.0,1.331604e+02,1.060823,0.559841,0.885996,0.333743,0.650913,6.994822e-01,...,0.867521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11786,Common_Yellowthroat_0125_190902.jpg,199,386.0,9.101219e+01,1.442437,0.493606,0.863535,0.720677,1.322599,5.855958e-01,...,0.958904,41.999992,1.25,0.02381,NaN,NaN,0.05814,17.199997,NaN,NaN


In [10]:
# Get the 'image' column for joining
train_df['image'] = train_df['filepath'].str.split('/').str[1]
val_df['image'] = val_df['filepath'].str.split('/').str[1]
test_df['image'] = test_df['filepath'].str.split('/').str[1]

# join the datasets
train_df = pd.merge(train_df, all_feats_df, on='image', how='left')
val_df = pd.merge(val_df, all_feats_df, on='image', how='left')
test_df = pd.merge(test_df, all_feats_df, on='image', how='left')

train_df

,img_id,filepath,label,is_train,image,class_id,part1_area,part1_perimeter,part1_aspect_ratio,part1_extent,...,area_ratio_part9_to_part4,area_ratio_part9_to_part10,area_ratio_part10_to_part8,area_ratio_part10_to_part9,area_ratio_part3_to_part8,area_ratio_part8_to_part3,area_ratio_part10_to_part11,area_ratio_part11_to_part10,area_ratio_part3_to_part9,area_ratio_part9_to_part3
0,4872,084.Red_legged_Kittiwake/Red_Legged_Kittiwake_...,83,0,Red_Legged_Kittiwake_0039_795429.jpg,83,3232.0,522.345215,1.851210,0.248768,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4017,070.Green_Violetear/Green_Violetear_0041_79564...,69,1,Green_Violetear_0041_795648.jpg,69,1044.0,226.503571,6.656366,0.444255,...,NaN,NaN,0.666666,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1794,032.Mangrove_Cuckoo/Mangrove_Cuckoo_0005_79459...,31,1,Mangrove_Cuckoo_0005_794599.jpg,31,1627.0,211.580734,2.502074,0.505908,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,6522,112.Great_Grey_Shrike/Great_Grey_Shrike_0010_7...,111,0,Great_Grey_Shrike_0010_797023.jpg,111,91.0,58.071068,7.359472,0.541667,...,0.171521,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2341,041.Scissor_tailed_Flycatcher/Scissor_Tailed_F...,40,0,Scissor_Tailed_Flycatcher_0058_41948.jpg,40,630.0,112.083260,2.494917,0.596591,...,0.463415,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8836,2936,051.Horned_Grebe/Horned_Grebe_0064_35015.jpg,50,0,Horned_Grebe_0064_35015.jpg,50,2426.0,246.166519,3.112218,0.693143,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8837,11511,196.House_Wren/House_Wren_0134_188077.jpg,195,1,House_Wren_0134_188077.jpg,195,6.0,5.414214,1.732051,0.666667,...,0.013699,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8838,578,011.Rusty_Blackbird/Rusty_Blackbird_0052_7035.jpg,10,0,Rusty_Blackbird_0052_7035.jpg,10,1292.0,220.965515,2.215001,0.376457,...,NaN,NaN,NaN,NaN,NaN,NaN,1.421052,0.703704,NaN,NaN
8839,11494,196.House_Wren/House_Wren_0061_187911.jpg,195,0,House_Wren_0061_187911.jpg,195,13675.0,532.658936,1.293826,0.616074,...,0.292636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
val_df

,img_id,filepath,label,is_train,image,class_id,part1_area,part1_perimeter,part1_aspect_ratio,part1_extent,...,area_ratio_part9_to_part4,area_ratio_part9_to_part10,area_ratio_part10_to_part8,area_ratio_part10_to_part9,area_ratio_part3_to_part8,area_ratio_part8_to_part3,area_ratio_part10_to_part11,area_ratio_part11_to_part10,area_ratio_part3_to_part9,area_ratio_part9_to_part3
0,4933,085.Horned_Lark/Horned_Lark_0076_73931.jpg,84,1,Horned_Lark_0076_73931.jpg,84,1924.0,216.823380,2.455318,0.516649,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,4440,077.Tropical_Kingbird/Tropical_Kingbird_0081_6...,76,1,Tropical_Kingbird_0081_69656.jpg,76,3934.0,340.421356,2.596235,0.605603,...,1.523365,5.620690,0.483333,0.177914,NaN,NaN,NaN,NaN,NaN,NaN
2,8953,153.Philadelphia_Vireo/Philadelphia_Vireo_0001...,152,1,Philadelphia_Vireo_0001_156554.jpg,152,7780.0,439.002106,1.754273,0.518252,...,NaN,NaN,5.447369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3170,055.Evening_Grosbeak/Evening_Grosbeak_0025_372...,54,0,Evening_Grosbeak_0025_37230.jpg,54,2446.0,385.102600,1.915485,0.268791,...,0.230769,1.499999,0.006689,0.666666,NaN,NaN,NaN,NaN,NaN,NaN
4,3522,061.Heermann_Gull/Heermann_Gull_0090_45834.jpg,60,0,Heermann_Gull_0090_45834.jpg,60,2803.0,298.285309,1.976887,0.536049,...,0.588710,NaN,NaN,NaN,6.693548,0.149398,NaN,NaN,11.369863,0.087952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1763,10028,171.Myrtle_Warbler/Myrtle_Warbler_0054_166985.jpg,170,0,Myrtle_Warbler_0054_166985.jpg,170,2369.0,259.651794,1.358569,0.484359,...,0.128205,0.029586,NaN,33.799995,NaN,NaN,NaN,NaN,NaN,NaN
1764,7551,129.Song_Sparrow/Song_Sparrow_0036_121679.jpg,128,1,Song_Sparrow_0036_121679.jpg,128,5031.0,411.255890,2.624046,0.496154,...,0.562358,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1765,2952,052.Pied_billed_Grebe/Pied_Billed_Grebe_0037_3...,51,0,Pied_Billed_Grebe_0037_35598.jpg,51,2522.0,313.628448,2.835750,0.609768,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1766,3480,060.Glaucous_winged_Gull/Glaucous_Winged_Gull_...,59,0,Glaucous_Winged_Gull_0013_44381.jpg,59,1305.0,307.764496,2.442199,0.226287,...,2.198795,NaN,NaN,NaN,24.666664,0.040541,NaN,NaN,0.608219,1.644144


In [12]:
test_df

,img_id,filepath,label,is_train,image,class_id,part1_area,part1_perimeter,part1_aspect_ratio,part1_extent,...,area_ratio_part9_to_part4,area_ratio_part9_to_part10,area_ratio_part10_to_part8,area_ratio_part10_to_part9,area_ratio_part3_to_part8,area_ratio_part8_to_part3,area_ratio_part10_to_part11,area_ratio_part11_to_part10,area_ratio_part3_to_part9,area_ratio_part9_to_part3
0,3503,061.Heermann_Gull/Heermann_Gull_0077_45711.jpg,60,1,Heermann_Gull_0077_45711.jpg,60,1.0,1.000000e-06,0.000000,1.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8509,145.Elegant_Tern/Elegant_Tern_0044_150946.jpg,144,0,Elegant_Tern_0044_150946.jpg,144,5.0,3.828427e+00,8.660254,0.250000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8589,147.Least_Tern/Least_Tern_0036_153658.jpg,146,1,Least_Tern_0036_153658.jpg,146,3328.0,5.021676e+02,1.926722,0.278261,...,1.252907,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,329,007.Parakeet_Auklet/Parakeet_Auklet_0072_79592...,6,0,Parakeet_Auklet_0072_795929.jpg,6,4284.0,5.406884e+02,1.065557,0.490160,...,0.563636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7.967742,0.125506
4,10948,186.Cedar_Waxwing/Cedar_Waxwing_0092_179123.jpg,185,1,Cedar_Waxwing_0092_179123.jpg,185,5402.0,4.246762e+02,2.077280,0.598361,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1174,8056,138.Tree_Swallow/Tree_Swallow_0092_136236.jpg,137,0,Tree_Swallow_0092_136236.jpg,137,263.0,1.140122e+02,5.686695,0.343791,...,NaN,NaN,29.99999,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1175,3156,055.Evening_Grosbeak/Evening_Grosbeak_0078_380...,54,0,Evening_Grosbeak_0078_38051.jpg,54,298.0,8.456854e+01,2.982295,0.380102,...,NaN,NaN,NaN,NaN,34.714279,0.028807,NaN,NaN,6.394737,0.156379
1176,6241,107.Common_Raven/Common_Raven_0128_102017.jpg,106,0,Common_Raven_0128_102017.jpg,106,7672.0,7.463290e+02,3.086730,0.262632,...,0.376033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1177,5636,097.Orchard_Oriole/Orchard_Oriole_0042_91678.jpg,96,1,Orchard_Oriole_0042_91678.jpg,96,28.0,1.848528e+01,1.902564,0.666667,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# ======================================================
# Helper: Load shape features
# ======================================================
def extract_shape_features(df):
    df = df.drop(columns=["image"])
    df = df.drop(columns=full_df.columns)
    df = df.filter(like='full_', axis=1)
    df = df.fillna(df.mean())
    # labels = torch.tensor(df["label"].values, dtype=torch.long)
    # Drop non-feature columns: assuming CSV columns are ['image', 'label', ...features]
    feats = df.to_numpy(dtype=np.float32)
    feats_tensor = torch.tensor(feats, dtype=torch.float32)
    return feats_tensor #, labels

In [14]:
train_feats = extract_shape_features(train_df)
val_feats = extract_shape_features(val_df)
test_feats = extract_shape_features(test_df)

train_mean = train_feats.mean(dim=0)
train_std = train_feats.std(dim=0) + 1e-6

train_feats = (train_feats - train_mean) / train_std
val_feats = (val_feats - train_mean) / train_std
test_feats = (test_feats - train_mean) / train_std

FEATURE_SIZE = train_feats.shape[1]
print(f"Train feats: {train_feats.shape}, Val feats: {val_feats.shape}, Test feats: {test_feats.shape}")
print(f"number of shape features:", FEATURE_SIZE)

Train feats: torch.Size([8841, 233]), Val feats: torch.Size([1768, 233]), Test feats: torch.Size([1179, 233])
number of shape features: 233


In [15]:
class ShapeEncoder(nn.Module):
    def __init__(self, embed_dim=256):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.fc = nn.Linear(128, embed_dim)

    def forward(self, mask_tensor):
        """
        mask_tensor: (B, 3, 64, 64)
        """
        feat = self.encoder(mask_tensor)
        feat = feat.flatten(1)
        z = self.fc(feat)
        return z  

class GatedFusion(nn.Module):
    def __init__(self, img_dim, shape_dim):
        super().__init__()

        self.shape_proj = nn.Linear(shape_dim, img_dim)

        self.gate = nn.Sequential(
            nn.Linear(img_dim + img_dim, img_dim),
            nn.LayerNorm(img_dim),
            nn.Sigmoid()
        )

    def forward(self, z_img, z_shape):
        z_shape_proj = self.shape_proj(z_shape)

        concat = torch.cat([z_img, z_shape_proj], dim=1)
        g = self.gate(concat)

        fused = z_img + g * (z_shape_proj - z_img)
        return fused

class DeepShapeFusionModel(nn.Module):
    def __init__(self, num_classes, shape_embed_dim=256):
        super().__init__()
        self.num_shape_feats = FEATURE_SIZE

        # Modules
        self.seg_module = load_segmentation(
            name=SEGMENTATION_NAME, 
            encoder_name=SEGMENTATION_ENCODER, 
            checkpoint_path=SEGMENTATION_WEIGHTS
        )
        self.backbone = timm.create_model(model_name=CLASSIFIER_NAME, pretrained=True, num_classes=0)
        self.shape_encoder = ShapeEncoder(shape_embed_dim)

        self.fusion = GatedFusion(
            img_dim=self.backbone.num_features,
            shape_dim=shape_embed_dim
        )

        total_feats = self.backbone.num_features + self.num_shape_feats # + shape_embed_dim

        self.classifier = nn.Sequential(
                                nn.LayerNorm(total_feats),
                                nn.Dropout(0.3),
                                nn.Linear(total_feats, num_classes)
                            )

        for p in self.seg_module.parameters():
            p.requires_grad = False
        self.seg_module.eval()

    def forward(self, x, shape_feats=None):

        # ---- Segmentation ----
        with torch.no_grad():
            mask_logits = self.seg_module(x)
            masks = torch.argmax(mask_logits, dim=1)  # (B, H, W) with labels 0,1,2,3
        # mask_logits = self.seg_module(x)
        # mask_probs = torch.softmax(mask_logits, dim=1)  # example single part

        # Extract part embeddings
        parts = mask_logits[:, 1:, :, :]  # remove background, will try mask_probs as well
        mask_binary = parts.sum(dim=1, keepdim=True)
        edge = torch.abs(
            torch.nn.functional.avg_pool2d(mask_binary, 3, stride=1, padding=1)
            - mask_binary
        )
        shape_tensor = torch.cat([mask_binary, edge, mask_binary], dim=1)
        z_shape = self.shape_encoder(shape_tensor)

        # ---- Extract hand-crafted shape descriptors ----
        # ---- Shape features ----
        if shape_feats is None:
            batch_size = x.size(0)
            batch_feats = []
            for i in range(batch_size):
                img_np = x[i].detach().cpu().numpy().transpose(1, 2, 0)
                mask_np = masks[i].detach().cpu().numpy()
                feats_dict = extract_all_features(img_np, mask_np)
                # Convert dict to tensor (relies on consistent insertion order for 937 features)
                feat_tensor = torch.tensor(list(feats_dict.values()), dtype=torch.float32)
                batch_feats.append(feat_tensor)
            shape_feats = torch.stack(batch_feats).to(x.device)  # (B, 937)
        # shape_feats = torch.tensor(shape_feats).to(x.device)        

        # ---- Image Encoding ----
        z_img = self.backbone(x)
        fused = self.fusion(z_img, z_shape)
        combined = torch.cat([fused, shape_feats], dim=1)  # (B, total_feats)

        # ---- Classification ----
        out = self.classifier(combined)
        return out, mask_logits, z_shape


In [16]:
device = torch.device(f"cuda:{DEVICE_ID}")
model = DeepShapeFusionModel(num_classes=num_classes, shape_embed_dim=512)
model.to(device)

DeepShapeFusionModel(
  (seg_module): DeepLabV3Plus(
    (encoder): ResNetEncoder(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (bn2): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(128, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_runni

In [17]:
# Setting up the criterion, optimizer and scheduler
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)  #label_smoothing=0.1)
if OPTIMIZER == "SGD":
    optimizer = torch.optim.SGD(
        model.parameters(), 
        lr=LR, 
        momentum=0.9, 
        weight_decay=1e-4)
elif OPTIMIZER == "AdamW":
    optimizer = torch.optim.AdamW(
        model.parameters(), 
        lr=LR, 
        weight_decay=1e-4)
else:
    optimizer = torch.optim.RMSprop(
        model.parameters(),
        lr=LR,       # typically around 0.256 / batch_size scaling, you can tune
        alpha=0.9,              # decay (smoothing constant)
        eps=0.001,              # numerical stability
        momentum=0.9,           # as in paper
        weight_decay=1e-5,      # L2 regularization
        centered=False          # same as paper’s setup
    )
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=2)
scaler = torch.amp.GradScaler('cuda')

In [18]:
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0, 0, 0
    pbar = tqdm(train_loader, desc="[Train]", position=0, leave=False)
    b_pos = 0 # Batch position
    for batch_idx, (images, labels, indices) in enumerate(pbar):
        images, labels = images.to(device), labels.to(device)        
        # shape_feats_batch = train_feats[b_pos:b_pos+images.size(0)].to(device)
        shape_feats_batch = train_feats[indices].to(device)
        b_pos += images.size(0)

        optimizer.zero_grad()
        outputs, _, _ = model(images, shape_feats_batch)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

        pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy


def validate_one_epoch(model, val_loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    b_pos = 0 # Batch position
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="[Val]", position=0, leave=False)
        for batch_idx, (images, labels, indices) in enumerate(pbar):
            images, labels = images.to(device), labels.to(device)
            # shape_feats_batch = val_feats[b_pos:b_pos+images.size(0)].to(device)
            shape_feats_batch = val_feats[indices].to(device)
            b_pos += images.size(0)

            # optimizer.zero_grad()
            outputs, _, _ = model(images, shape_feats_batch)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix(loss=loss.item())
    pbar.close()
    avg_loss = running_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                device, epochs, log_dir):
    # Full training loop with logging and checkpointing 
    history = {"epoch": [], "train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    checkpoints_dir = os.path.join(log_dir, "checkpoints")
    os.makedirs(checkpoints_dir, exist_ok=True)

    # Initialize validation loss and start training
    best_val_loss = float("inf")
    print("Starting the first epoch...")

    for epoch in range(epochs):
        # ---- Training ----
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)

        # ---- Validation ----
        val_loss, val_acc = validate_one_epoch(model, val_loader, criterion, device)

        # ---- Scheduler ----
        scheduler.step(val_loss)

        # ---- Logging ----
        history["epoch"].append(epoch + 1)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        # ---- Checkpoint ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = os.path.join(checkpoints_dir, "best_model.pth")
            torch.save(model.state_dict(), best_path)

            checkpoint_path = os.path.join(checkpoints_dir, f"checkpoint_epoch{epoch+1}.pth")
            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_loss
            }, checkpoint_path)
        
        print(f"\r Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")    
    print("Training complete!")

    # ---- Save logs ----
    df = pd.DataFrame(history)
    csv_logfile = os.path.join(log_dir, "training_log.csv")
    json_logfile = os.path.join(log_dir, "training_log.json")

    df.to_csv(csv_logfile, index=False)
    with open(json_logfile, "w") as f:
        json.dump(history, f, indent=4)

    return history

In [19]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    device=device,
    epochs=EPOCHS,
    log_dir=LOG_DIR
)

Starting the first epoch...


 Epoch 1/100 | Train Loss: 3.9930 | Train Acc: 0.2991 | Val Loss: 2.5855 | Val Acc: 0.6516


 Epoch 2/100 | Train Loss: 2.0327 | Train Acc: 0.7536 | Val Loss: 1.7584 | Val Acc: 0.7743


 Epoch 3/100 | Train Loss: 1.4646 | Train Acc: 0.8646 | Val Loss: 1.5494 | Val Acc: 0.8179


 Epoch 4/100 | Train Loss: 1.2492 | Train Acc: 0.9179 | Val Loss: 1.4682 | Val Acc: 0.8247


...

 Epoch 33/100 | Train Loss: 0.8977 | Train Acc: 1.0000 | Val Loss: 1.4340 | Val Acc: 0.8558


 Epoch 34/100 | Train Loss: 0.8978 | Train Acc: 1.0000 | Val Loss: 1.4339 | Val Acc: 0.8563


 Epoch 35/100 | Train Loss: 0.8984 | Train Acc: 1.0000 | Val Loss: 1.4334 | Val Acc: 0.8569


 Epoch 36/100 | Train Loss: 0.8978 | Train Acc: 1.0000 | Val Loss: 1.4336 | Val Acc: 0.8563


KeyboardInterrupt: 

In [20]:
test_loss, test_acc = validate_one_epoch(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

Test Loss: 1.4661 | Test Acc: 0.8516
